# Trabajo Práctico Integrador - Introducción al Análisis de Datos

**Caso:** análisis y predicción de cancelaciones en reservas hoteleras.

> La empresa busca comprender qué factores influyen en la cancelación de reservas en hoteles urbanos y resort, analizando variables como fechas de estadía, tipo de cliente, canal de reserva, historial previo y tarifas, con el fin de identificar patrones y mejorar la gestión operativa.


## Desarrollado por

- **Apellido y nombre:** Matias Carro
- **Comisión:** 11
- **Dataset asignado:** Dataset K
- **Entrega:** Primer Entrega - Semana 3
- **Fecha:** 

## Importación de librerías

### Librerias a utilizar:

- **pandas:** para cargar el dataset y trabajar con datos en forma de tablas (DataFrames).
- **numpy:** para realizar operaciones numéricas y manejar arreglos de forma eficiente.



In [25]:
import pandas as pd
import numpy as np

print("Versión de Pandas:", pd.__version__)
print("Versión de numpy:", np.__version__)
print("\nLibrerías cargadas correctamente.")


Versión de Pandas: 3.0.5
Versión de numpy: 2.5.2

Librerías cargadas correctamente.


## 1. Presentacion del problema

El caso de estudio se centra en el análisis de reservas hoteleras con el objetivo de comprender qué factores están asociados a la cancelación de estadías. El dataset asignado contiene información detallada de cada reserva, incluyendo tipo de hotel, fechas de llegada, duración de la estadía, composición del grupo, país de origen, canal de reserva, tipo de cliente, historial previo, tarifa promedio por noche y características operativas como depósito, agente, pedidos especiales y cambios realizados.


### Relación entre datos, información y conocimiento
En este trabajo partimos de los **datos** que son valores crudos del sistema de reservas: fechas, cantidades, categorías y códigos.  
Mediante el análisis exploratorio estos datos se convierten en **información**, como distribuciones, patrones y diferencias entre reservas canceladas y no canceladas.  
A partir de esa información generamos el **conocimiento** que nos permite entender el comportamiento de los clientes y detectar factores que podrían influir en la cancelación de una reserva.  

Esta relación es clave para el caso: los datos del hotel por sí solos no dicen nada, pero al transformarlos en información y luego interpretarlos, podemos identificar variables relevantes (como `lead_time`, `deposit_type` o `customer_type`) que ayudan a explicar por qué algunas reservas se cancelan y otras no.

### Ciclo de vida del análisis
Este trabajo se enmarca en el ciclo de vida del análisis de datos, que incluye:
1. Obtención del dataset asignado.  
2. Comprensión inicial del problema (cancelaciones hoteleras).  
3. Exploración y limpieza mínima (EDA).  
4. Transformación y preparación de variables relevantes.  
5. Interpretación y comunicación de resultados.

### Variable objetivo
La **variable objetivo** del análisis es **`is_canceled`**, que indica si la reserva fue cancelada (`1`) o no (`0`).  
Su distribución será calculada y analizada en las próximas secciones para comprender el comportamiento general del conjunto de datos y orientar las preguntas del análisis.

### Preguntas iniciales que orientan el trabajo
- ¿Qué características diferencian a las reservas canceladas de las no canceladas?  
- ¿Influyen el tipo de hotel o el canal de reserva en la cancelación?  
- ¿Las reservas con mayor anticipación (`lead_time`) presentan mayor probabilidad de cancelación?  
- ¿Los clientes con pedidos especiales o estacionamiento tienden a cancelar menos?  
- ¿Las políticas de depósito (`deposit_type`) reducen la cancelación?  
- ¿Existen segmentos de mercado con mayor riesgo de cancelación?  

## 2. Carga del dataset

Se carga el Dataset perteneciente a la comisión 11:


In [26]:
df = pd.read_csv("hotel booking TPI grupo K.csv")

print("Dataset cargado correctamente.")
print("\nVista de los primeros datos: ")
df.head()


Dataset cargado correctamente.

Vista de los primeros datos: 


,booking_id,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,arrival_date,stays_in_weekend_nights,...,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests
0,HB-014269,Resort Hotel,0,17,2025,February,8,21,2025-02-21,0,...,E,0,No Deposit,NaN,292.0,0,Transient,58.00,1,1
1,HB-018087,Resort Hotel,0,6,2023,November,44,1,2023-11-01,2,...,D,3,No Deposit,240.0,NaN,0,Transient,58.00,0,2
2,HB-022870,Resort Hotel,0,45,2024,April,15,8,2024-04-08,0,...,D,1,No Deposit,240.0,NaN,0,Transient-Party,65.00,0,2
3,HB-048154,City Hotel,0,95,2024,March,11,17,2024-03-17,2,...,A,0,No Deposit,9.0,NaN,0,Transient,73.95,0,1
4,HB-060351,City Hotel,1,277,2024,November,45,7,2024-11-07,1,...,A,0,Non Refund,NaN,NaN,0,Transient,100.00,0,0


## 4. Estructura general:

Filas y Columnas:

In [27]:
filas, columnas = df.shape
print(f"El dataset tiene {filas} filas y {columnas} columnas.")

print("Columnas del dataset:")
print(df.columns.tolist())

print("\nTipos de datos:")
print(df.info())


El dataset tiene 25000 filas y 32 columnas.
Columnas del dataset:
['booking_id', 'hotel', 'is_canceled', 'lead_time', 'arrival_date_year', 'arrival_date_month', 'arrival_date_week_number', 'arrival_date_day_of_month', 'arrival_date', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'meal', 'country', 'market_segment', 'distribution_channel', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'reserved_room_type', 'assigned_room_type', 'booking_changes', 'deposit_type', 'agent', 'company', 'days_in_waiting_list', 'customer_type', 'adr', 'required_car_parking_spaces', 'total_of_special_requests']

Tipos de datos:
<class 'pandas.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 32 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   booking_id                      25000 non-null  str    
 1   hotel                          

El dataset tiene 25.000 filas y 32 columnas.  
Las variables incluyen información temporal, categórica y numérica relevante para el análisis.


### Diccionario de variables 

| Variable | Traducción | Representación | Tipo de Datos |
|----------|------------|----------------|-----------|
| booking_id | ID de reserva | Identificador único de cada reserva. | str |
| hotel | Tipo de hotel | Indica si la reserva corresponde a City Hotel (Ciudad) o Resort Hotel (Resort). | str |
| is_canceled | Cancelada | Indica si la reserva fue cancelada (1) o no (0). | int64 |
| lead_time | Anticipación | Días entre la fecha de reserva y la fecha de llegada. | int64 |
| arrival_date_year | Año de llegada | Año en el que el huésped llega al hotel. | int64 |
| arrival_date_month | Mes de llegada | Mes en el que el huésped llega al hotel. | str |
| arrival_date_week_number | Semana de llegada | Número de semana del año en la que llega el huésped. | int64 |
| arrival_date_day_of_month | Día del mes de llegada | Día del mes en el que llega el huésped. | int64 |
| arrival_date | Fecha de llegada | Fecha completa de llegada (YYYY-MM-DD). | str |
| stays_in_weekend_nights | Noches de fin de semana | Cantidad de noches en fines de semana. | int64 |
| stays_in_week_nights | Noches de semana | Cantidad de noches de lunes a jueves. | int64 |
| adults | Adultos | Número de adultos en la reserva. | int64 |
| children | Niños | Número de niños en la reserva. | float64 |
| babies | Bebés | Número de bebés en la reserva. | int64 |
| meal | Tipo de comida | Plan de comidas asociado a la reserva (BB, HB, SC, etc.). | str |
| country | País | País de origen del huésped. | str |
| market_segment | Segmento de mercado | Tipo de cliente según el canal de adquisición. | str |
| distribution_channel | Canal de distribución | Canal por el cual se realizó la reserva. | str |
| is_repeated_guest | Huésped repetido | Indica si el cliente ya se alojó anteriormente. | int64 |
| previous_cancellations | Cancelaciones previas | Cantidad de reservas previas canceladas por el cliente. | int64 |
| previous_bookings_not_canceled | Reservas previas no canceladas | Cantidad de reservas previas completadas por el cliente. | int64 |
| reserved_room_type | Habitación reservada | Tipo de habitación solicitada originalmente. | str |
| assigned_room_type | Habitación asignada | Tipo de habitación finalmente asignada. | str |
| booking_changes | Cambios en la reserva | Número de modificaciones realizadas a la reserva. | int64 |
| deposit_type | Tipo de depósito | Política de depósito aplicada. | str |
| agent | Agente | Código del agente que gestionó la reserva. | float64 |
| company | Compañía | Código de la empresa asociada a la reserva. | float64 |
| days_in_waiting_list | Días en lista de espera | Tiempo que la reserva permaneció en espera antes de confirmarse. | int64 |
| customer_type | Tipo de cliente | Clasificación del cliente (Transient, Contract, Group, etc.). | str |
| adr | Tarifa promedio diaria | Precio promedio por noche de la reserva. | float64 |
| required_car_parking_spaces | Estacionamiento requerido | Cantidad de espacios de estacionamiento solicitados. | int64 |
| total_of_special_requests | Pedidos especiales | Número de solicitudes especiales realizadas por el cliente. | int64 |



## Tipos de variables 

Clasificacion de las variables por su tipo de datos

In [28]:
numericas = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
cadenas = df.select_dtypes(include=['str']).columns.tolist()

print("Cantidad de variables numéricas:", len(numericas))
print("Cantidad de variables de texto / fecha:", len(cadenas))



Cantidad de variables numéricas: 20
Cantidad de variables de texto / fecha: 12


### Clasificación inicial de las variables

| Variable | Tipo | Representación |
|----------|------|----------------|
| booking_id | Categórica (ID) | Identificador único de reserva. |
| hotel | Categórica | Tipo de hotel. |
| is_canceled | Numérica (booleana) | Indica si la reserva fue cancelada. |
| lead_time | Numérica | Días de anticipación de la reserva. |
| arrival_date_year | Numérica | Año de llegada. |
| arrival_date_month | Categórica | Mes de llegada. |
| arrival_date_week_number | Numérica | Semana del año. |
| arrival_date_day_of_month | Numérica | Día del mes. |
| arrival_date | Temporal | Fecha completa de llegada. |
| stays_in_weekend_nights | Numérica | Noches de fin de semana. |
| stays_in_week_nights | Numérica | Noches de semana. |
| adults | Numérica | Cantidad de adultos. |
| children | Numérica | Cantidad de niños. |
| babies | Numérica | Cantidad de bebés. |
| meal | Categórica | Tipo de comida. |
| country | Categórica | País de origen. |
| market_segment | Categórica | Segmento de mercado. |
| distribution_channel | Categórica | Canal de distribución. |
| is_repeated_guest | Numérica (booleana) | Indica si el huésped ya se alojó antes. |
| previous_cancellations | Numérica | Cancelaciones previas. |
| previous_bookings_not_canceled | Numérica | Reservas previas no canceladas. |
| reserved_room_type | Categórica | Habitación reservada. |
| assigned_room_type | Categórica | Habitación asignada. |
| booking_changes | Numérica | Cambios realizados a la reserva. |
| deposit_type | Categórica | Política de depósito. |
| agent | Categórica (ID) | Código del agente. |
| company | Categórica (ID) | Código de la compañía. |
| days_in_waiting_list | Numérica | Días en lista de espera. |
| customer_type | Categórica | Tipo de cliente. |
| adr | Numérica | Tarifa promedio diaria. |
| required_car_parking_spaces | Numérica | Espacios de estacionamiento solicitados. |
| total_of_special_requests | Numérica | Cantidad de pedidos especiales. |

---


### Resumen descriptivo

In [29]:
df.describe().round(2)

,is_canceled,lead_time,arrival_date_year,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,booking_changes,agent,company,days_in_waiting_list,adr,required_car_parking_spaces,total_of_special_requests
count,25000.00,25000.00,25000.00,25000.00,25000.00,25000.00,25000.00,25000.00,25000.00,25000.00,25000.00,25000.00,25000.00,25000.00,21511.00,1426.00,25000.00,25000.00,25000.00,25000.00
mean,0.37,103.29,2024.16,26.51,15.83,0.92,2.48,1.85,0.10,0.01,0.03,0.08,0.13,0.22,86.05,189.79,2.40,101.86,0.06,0.57
std,0.48,106.59,0.71,13.40,8.81,0.99,1.88,0.58,0.39,0.12,0.18,0.79,1.43,0.63,110.38,132.42,17.56,48.03,0.24,0.80
min,0.00,0.00,2023.00,1.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,9.00,0.00,-6.38,0.00,0.00
25%,0.00,18.00,2024.00,16.00,8.00,0.00,1.00,2.00,0.00,0.00,0.00,0.00,0.00,0.00,9.00,62.00,0.00,69.50,0.00,0.00
50%,0.00,68.00,2024.00,27.00,16.00,1.00,2.00,2.00,0.00,0.00,0.00,0.00,0.00,0.00,14.00,174.00,0.00,95.00,0.00,0.00
75%,1.00,159.00,2025.00,37.00,24.00,2.00,3.00,2.00,0.00,0.00,0.00,0.00,0.00,0.00,229.00,277.00,0.00,126.00,0.00,1.00
max,1.00,629.00,2025.00,52.00,31.00,14.00,34.00,50.00,3.00,10.00,1.00,26.00,66.00,14.00,531.00,539.00,391.00,510.00,3.00,5.00


## Caracterización descriptiva inicial:


### 1. booking_id


In [30]:
ids_unicos = df["booking_id"].nunique()
faltantes = df["booking_id"].isnull().sum()
porcentaje = (faltantes / len(df)) * 100
print ("Se trata de", ids_unicos,"IDs unicas")
print("Cantidad de faltantes:" , faltantes)
print(porcentaje, "% de variables faltantes")

print("\n Muestra:")
df["booking_id"].value_counts().head()

Se trata de 25000 IDs unicas
Cantidad de faltantes: 0
0.0 % de variables faltantes

 Muestra:


booking_id
HB-014269    1
HB-018087    1
HB-022870    1
HB-048154    1
HB-060351    1
Name: count, dtype: int64

**Descripción:**  
Es un identificador único, no aporta información estadística. Se usa solo para referencia.

**Cantidad y Faltantes:** 
Se cuenta con 25000 IDs unicas, sin ninguna faltante

---

### 2. hotel

In [31]:

conteo_total = df["hotel"].count()

faltantes = df["hotel"].isnull().sum()
porcentaje_faltantes = (faltantes / len(df)) * 100

distintos = df["hotel"].nunique()

conteo = df["hotel"].value_counts()
porcentaje = df["hotel"].value_counts(normalize=True) * 100

print("Datos totales:", conteo_total)
print("Faltantes:", faltantes)
print("Porcentaje de faltantes:", porcentaje_faltantes, "%")

print("Cantidad de valores distintos:", distintos)

print("\nDistribución:")
print(conteo)

print("\nPorcentaje:")
print(porcentaje)

print("\nMuestra:")
print(df["hotel"].value_counts().head())


Datos totales: 25000
Faltantes: 0
Porcentaje de faltantes: 0.0 %
Cantidad de valores distintos: 2

Distribución:
hotel
City Hotel      16716
Resort Hotel     8284
Name: count, dtype: int64

Porcentaje:
hotel
City Hotel      66.864
Resort Hotel    33.136
Name: proportion, dtype: float64

Muestra:
hotel
City Hotel      16716
Resort Hotel     8284
Name: count, dtype: int64


**Descripción:**  
La variable `hotel` indica el tipo de establecimiento donde se realizó la reserva. Es útil para comparar comportamientos entre City Hotel y Resort Hotel, ya que cada uno puede tener patrones diferentes de demanda, estacionalidad y cancelaciones.

**Faltantes:**  
0 (0%)

**Valores distintos:**  
2

**Distribución:**  
- City Hotel: 67%  
- Resort Hotel: 33%

**Observación:**  
La mayoría de las reservas corresponden al City Hotel. 

---

### 3. is_canceled

In [32]:
conteo_total = df["is_canceled"].count()

faltantes = df["is_canceled"].isnull().sum()
porcentaje_faltantes = (faltantes / len(df)) * 100

distintos = df["is_canceled"].nunique()

conteo = df["is_canceled"].value_counts()
porcentaje = df["is_canceled"].value_counts(normalize=True) * 100

print("Datos totales:", conteo_total)
print("Faltantes:", faltantes)
print("Porcentaje de faltantes:", porcentaje_faltantes, "%")

print("Cantidad de valores distintos:", distintos)

print("\nDistribución:")
print(conteo)

print("\nPorcentaje:")
print(porcentaje)

print("\nMuestra:")
print(df["is_canceled"].value_counts().head())


Datos totales: 25000
Faltantes: 0
Porcentaje de faltantes: 0.0 %
Cantidad de valores distintos: 2

Distribución:
is_canceled
0    15729
1     9271
Name: count, dtype: int64

Porcentaje:
is_canceled
0    62.916
1    37.084
Name: proportion, dtype: float64

Muestra:
is_canceled
0    15729
1     9271
Name: count, dtype: int64


### 3. is_canceled

**Descripción:**  
La variable `is_canceled` indica si la reserva fue cancelada (`1`) o no (`0`). Es una variable clave porque representa el resultado final del proceso de reserva y suele ser la variable objetivo en modelos de predicción de cancelaciones.

**Faltantes:**  
0 (0%)

**Valores distintos:**  
2  
- `0`: reserva no cancelada  
- `1`: reserva cancelada  

**Distribución:**  
- No canceladas: ~60%  
- Canceladas: ~40%

**Observación:**  
El dataset presenta una proporción considerable de cancelaciones. Esta distribución es importante para evaluar y llegar a una conclusion del motivo tan alto de las cancelaciones.

---

### 4. lead_time

In [33]:

#agrupacion de los rangos
step = 50
bins = np.arange(0, df["lead_time"].max() + step, step)
lead_time_grupos = pd.cut(df["lead_time"], bins=bins, include_lowest=True)

conteo_total = df["lead_time"].count()
faltantes = df["lead_time"].isnull().sum()
porcentaje_faltantes = (faltantes / len(df)) * 100
distintos = df["lead_time"].nunique()

conteo = lead_time_grupos.value_counts()
porcentaje = lead_time_grupos.value_counts(normalize=True) * 100

print("\nDatos totales:", conteo_total)
print("Faltantes:", faltantes)
print("Porcentaje de faltantes:", porcentaje_faltantes, "%")
print("Cantidad de valores distintos:", distintos)

print("\nDistribución por rangos:")
print(conteo)

print("\nPorcentaje por rangos:")
print(porcentaje)

print("\nMuestra:")
print(lead_time_grupos.value_counts().head())






Datos totales: 25000
Faltantes: 0
Porcentaje de faltantes: 0.0 %
Cantidad de valores distintos: 460

Distribución por rangos:
lead_time
(-0.001, 50.0]    10700
(50.0, 100.0]      4460
(100.0, 150.0]     3093
(150.0, 200.0]     2480
(200.0, 250.0]     1426
(250.0, 300.0]     1196
(300.0, 350.0]      847
(350.0, 400.0]      360
(400.0, 450.0]      217
(450.0, 500.0]      126
(500.0, 550.0]       46
(600.0, 650.0]       29
(550.0, 600.0]       20
Name: count, dtype: int64

Porcentaje por rangos:
lead_time
(-0.001, 50.0]    42.800
(50.0, 100.0]     17.840
(100.0, 150.0]    12.372
(150.0, 200.0]     9.920
(200.0, 250.0]     5.704
(250.0, 300.0]     4.784
(300.0, 350.0]     3.388
(350.0, 400.0]     1.440
(400.0, 450.0]     0.868
(450.0, 500.0]     0.504
(500.0, 550.0]     0.184
(600.0, 650.0]     0.116
(550.0, 600.0]     0.080
Name: proportion, dtype: float64

Muestra:
lead_time
(-0.001, 50.0]    10700
(50.0, 100.0]      4460
(100.0, 150.0]     3093
(150.0, 200.0]     2480
(200.0, 250.0]   


**Descripción:**  
La variable lead_time representa la cantidad de días entre la fecha de reserva y la fecha de llegada (anticipacion). Para facilitar su interpretación, se agrupa en rangos de 50 días. 

**Faltantes:**  
0 (0%)

**Valores distintos:**  
460 valores diferentes.


**Distribución por rangos:**  
- 0–50 días: 42.8%  
- 50–100 días: 17.8%  
- 100–150 días: 12.4%  
- 150–200 días: 9.9%  
- 200–250 días: 5.7%  
- 250–300 días: 4.8%  
- 300–350 días: 3.4%  
- 350–400 días: 1.4%  
- 400–450 días: 0.9%  
- 450–500 días: 0.5%  
- 500–550 días: 0.18%  
- 550–600 días: 0.08%  
- 600–650 días: 0.12%

**Interpretación:**  
- **Casi la mitad de las reservas (42.8%) se realizan dentro de los primeros 50 días**, lo que muestra que no hay una gran anticipacion en la reserva de la mayoría de los clientes.  
- Entre **50 y 150 días** se encuentra otro grupo importante de reservas (aproximadamente 30%). 
- A partir de **150 días**, la frecuencia comienza a descender, mostrando que reservar con más de 5 meses de anticipación es menos común.  
- Los rangos superiores (más de 300 días) representan **menos del 6% del total**, mostrando que las reservas extremadamente anticipadas son casos excepcionales.  
- Los valores extremos (más de 500 días) son muy raros y pueden estar asociados a eventos especiales.

---


### 5. arrival_date_year


In [34]:
df["arrival_date_year"].value_counts()


arrival_date_year
2024    11906
2025     8533
2023     4561
Name: count, dtype: int64